# InceptionTime Multi-task Learning — Tiffany Flat 3-class

Multi-task model: InceptionTime nf=64 backbone with two output heads trained jointly.
- **Main task:** 3-class ECG classification (Control / MR / MS) — CrossEntropy loss
- **Auxiliary task:** Predict 6 standardized ECG interval values (rr_ms, rr_std_ms, p_dur_ms, qrs_dur_ms, qtc_ms, sokolow) — MSE regression loss

Total loss = CE + λ × MSE (λ=0.1)  
ECG intervals are **regression targets** (supervision signal), not model inputs — contrast with `incep_nf64_ecgiv` where intervals are input features.
At inference, only the classification head is used; the regression head is discarded.


In [ ]:
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"  # fix optree/_dynamo version conflict on Kaggle

from __future__ import annotations

import random
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
PROJECT_ROOT   = Path("/kaggle/input/datasets/pascalhafidzfajri/mlcv-mitral")
OUT_DIR        = Path(".")
NPZ_DIR        = PROJECT_ROOT / "mitral_images" / "mitral_images"
SPLIT_CSV_PATH = PROJECT_ROOT / "data_split_3level_exp10.csv"
INTERVALS_CSV  = PROJECT_ROOT / "ecg_intervals.csv"
ARTIFACT_DIR   = OUT_DIR / "dl_artifacts" / "inceptiontime_multitask_tiffany_flat"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES         = ["0_control", "1_significant_mr", "2_significant_ms"]
FLAT_LABEL_TO_CLASS = {"Kontrol": 0, "MR": 1, "MS": 2}
LEAD_ORDER          = ["I","II","III","aVR","aVL","aVF","V1","V2","V3","V4","V5","V6"]

# Interval columns — used as REGRESSION TARGETS (supervision), not model inputs
INTERVAL_COLS = ["rr_ms", "rr_std_ms", "p_dur_ms", "qrs_dur_ms", "qtc_ms", "sokolow"]
N_INTERVAL    = len(INTERVAL_COLS)  # 6

LAMBDA_AUX = 0.1  # weight of auxiliary interval regression loss: total = CE + LAMBDA_AUX * MSE

DEVICE         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IN_CHANNELS    = len(LEAD_ORDER)  # 12
BATCH_SIZE     = 32
EPOCHS         = 60
PATIENCE       = 15
LEARNING_RATE  = 1e-3
MIN_LR         = 1e-5
WEIGHT_DECAY   = 1e-4
GRAD_CLIP_NORM = 5.0
NUM_WORKERS    = 0
PRELOAD_TO_RAM       = True
SKIP_COMPLETED_FOLDS = True

AUGMENTATION_CONFIG = {
    "noise_prob": 0.50, "noise_std": 0.01,
    "scale_prob": 0.50, "scale_range": (0.95, 1.05),
    "time_mask_prob": 0.35, "time_mask_max_frac": 0.03,
    "lead_dropout_prob": 0.15, "max_dropped_leads": 1,
}

print('Device:', DEVICE)
print('INTERVALS_CSV exists:', INTERVALS_CSV.exists())
print(f'LAMBDA_AUX = {LAMBDA_AUX}')


## 1. Load and validate ECG intervals

In [ ]:
intervals_df = pd.read_csv(INTERVALS_CSV)
print(f"Intervals loaded: {len(intervals_df)} samples")
print(f"NaN rates: {intervals_df[INTERVAL_COLS].isna().mean().round(3).to_dict()}")

# Build stem → interval array lookup (NaN handled per-fold at training time)
INTERVAL_LOOKUP = dict(
    zip(intervals_df["stem"].astype(str),
        intervals_df[INTERVAL_COLS].values.astype(np.float32))
)
print(f"Lookup built: {len(INTERVAL_LOOKUP)} stems")

## 2. Data loading

In [ ]:
NPZ_RAM_CACHE: dict = {}

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def infer_original_code(stem):
    stem = str(stem).strip().upper()
    for p in ("BA","BB","A","B","C","D","E"):
        if stem.startswith(p): return p
    return "UNKNOWN"

def normalize_per_sample_per_lead(signal):
    finite = np.isfinite(signal)
    vc = np.maximum(finite.sum(axis=0, keepdims=True).astype(np.float32), 1.0)
    mu = np.where(finite, signal, 0.0).sum(axis=0, keepdims=True).astype(np.float32) / vc
    c  = np.where(finite, signal - mu, 0.0).astype(np.float32)
    std = np.sqrt(np.maximum((c**2).sum(axis=0, keepdims=True)/vc, 1e-6)).astype(np.float32)
    std = np.where(finite.sum(axis=0, keepdims=True)==0, 1.0, std)
    mu  = np.where(finite.sum(axis=0, keepdims=True)==0, 0.0, mu)
    return np.nan_to_num((np.where(finite, signal, mu) - mu) / std, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

def augment_signal_channels(signal_ch, rng, cfg):
    x = signal_ch.copy(); nl, ns = x.shape
    if rng.random() < cfg["scale_prob"]:
        lo, hi = cfg["scale_range"]; x *= rng.uniform(lo, hi, size=(nl,1)).astype(np.float32)
    if rng.random() < cfg["noise_prob"] and cfg["noise_std"] > 0:
        x += rng.normal(0.0, cfg["noise_std"], size=x.shape).astype(np.float32)
    if rng.random() < cfg["time_mask_prob"] and cfg["time_mask_max_frac"] > 0:
        mw = max(1, int(ns*cfg["time_mask_max_frac"]))
        w  = int(rng.integers(1, mw+1))
        s  = int(rng.integers(0, max(1, ns-w+1)))
        x[:, s:s+w] = 0.0
    if rng.random() < cfg["lead_dropout_prob"] and cfg["max_dropped_leads"] > 0:
        nd = int(rng.integers(1, cfg["max_dropped_leads"]+1))
        x[rng.choice(nl, size=nd, replace=False), :] = 0.0
    return x.astype(np.float32)

def load_npz_arrays(npz_path):
    with np.load(npz_path, allow_pickle=False) as d:
        return d["signal_matrix"].astype(np.float32), d["overlap_mask"].astype(np.uint8), d["interpolated_mask"].astype(np.uint8)

def get_npz_arrays(row):
    key = str(row["stem"])
    if not PRELOAD_TO_RAM: return load_npz_arrays(row["npz_path"])
    if key not in NPZ_RAM_CACHE: NPZ_RAM_CACHE[key] = load_npz_arrays(row["npz_path"])
    return NPZ_RAM_CACHE[key]

def preload_npz_frame(frame, desc):
    if not PRELOAD_TO_RAM: return
    pending = [(str(r["stem"]), str(r["npz_path"])) for _, r in frame.drop_duplicates("stem").iterrows()
               if str(r["stem"]) not in NPZ_RAM_CACHE]
    if not pending: return
    def load_one(item): return item[0], load_npz_arrays(item[1])
    with ThreadPoolExecutor(max_workers=4) as ex:
        for stem, arrays in tqdm((f.result() for f in as_completed(ex.submit(load_one,i) for i in pending)),
                                 total=len(pending), desc=f"preload {desc}", leave=False):
            NPZ_RAM_CACHE[stem] = arrays


class MitralECGDataset(Dataset):
    """Returns (signal, interval_vec, label, stem).
    interval_vec is pre-normalized by caller; NaN already imputed.
    """
    def __init__(self, frame, interval_means, interval_stds, augment=False, seed=SEED):
        self.frame = frame.reset_index(drop=True)
        self.means = interval_means   # (6,) np array
        self.stds  = interval_stds    # (6,) np array
        self.augment = augment
        self.seed = seed
        preload_npz_frame(self.frame, "ECG")

    def __len__(self): return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        stem = str(row["stem"])

        # Waveform
        signal, overlap, _ = get_npz_arrays(row)
        signal[overlap.astype(bool)] = 0.0
        signal_ch = normalize_per_sample_per_lead(signal).T  # (12, T)
        if self.augment:
            rng = np.random.default_rng(self.seed + idx*1009 + random.randint(0, 1_000_000))
            signal_ch = augment_signal_channels(signal_ch, rng, AUGMENTATION_CONFIG)

        # Interval features: lookup, impute NaN with train median, z-score
        raw = INTERVAL_LOOKUP.get(stem, np.full(N_INTERVAL, np.nan, dtype=np.float32))
        raw = np.where(np.isnan(raw), self.means, raw)   # impute
        ivec = ((raw - self.means) / np.maximum(self.stds, 1e-6)).astype(np.float32)

        return (
            torch.from_numpy(signal_ch),
            torch.from_numpy(ivec),
            torch.tensor(int(row["class_id"]), dtype=torch.long),
            stem,
        )


def compute_interval_stats(frame):
    """Compute per-feature median and IQR-based std from train frame stems."""
    stems = frame["stem"].astype(str).tolist()
    vecs = [INTERVAL_LOOKUP[s] for s in stems if s in INTERVAL_LOOKUP]
    arr = np.array(vecs, dtype=np.float32)  # (N, 6)
    means = np.nanmedian(arr, axis=0)
    stds  = np.nanstd(arr, axis=0)
    stds  = np.maximum(stds, 1e-6)
    return means.astype(np.float32), stds.astype(np.float32)


def load_tiffany_split():
    split = pd.read_csv(SPLIT_CSV_PATH)
    split["stem"] = split["filename"].map(lambda x: Path(str(x)).stem)
    split["original_code"] = split["stem"].map(infer_original_code)
    split["npz_path"] = split["stem"].map(lambda s: str(NPZ_DIR/f"{s}.npz"))
    split["npz_exists"] = split["npz_path"].map(lambda p: Path(p).exists())
    avail = split[split["npz_exists"]].copy().reset_index(drop=True)
    avail["class_id"] = avail["flat_label"].map(FLAT_LABEL_TO_CLASS)
    print(f"Available: {len(avail)}/{len(split)}")
    tv  = avail[avail["isHeldOut"]==0].copy().reset_index(drop=True)
    hld = avail[avail["isHeldOut"]==1].copy().reset_index(drop=True)
    print(f"Train/val: {len(tv)}, Heldout: {len(hld)}")
    return tv, hld


def make_loader(frame, means, stds, batch_size, shuffle, augment, seed):
    ds = MitralECGDataset(frame, means, stds, augment=augment, seed=seed)
    g  = torch.Generator().manual_seed(seed)
    return DataLoader(ds, batch_size, shuffle=shuffle, num_workers=NUM_WORKERS,
                      pin_memory=(DEVICE.type=="cuda"), generator=g)


set_seed(SEED)
train_val_df, heldout_df = load_tiffany_split()

## 3. Multi-task model architecture

```
Input: (B, 12, T) — ECG signal only
  └─ InceptionTime backbone (6 blocks, nf=64)
     └─ GAP → 256-dim embedding → Dropout(0.3)
         ├─ cls_head: Linear(256→3)  ← classification logits (main task)
         └─ reg_head: Linear(256→6)  ← z-scored interval predictions (auxiliary task)

Loss = CrossEntropy(cls) + 0.1 × MSE(reg, z_intervals)
Inference: cls_head only — reg_head discarded at test time.
```


In [ ]:
class InceptionBlock(nn.Module):
    def __init__(self, in_channels, n_filters=32, kernel_sizes=(40,20,10), bottleneck=True):
        super().__init__()
        bc = n_filters if bottleneck else in_channels
        self.bottleneck = nn.Conv1d(in_channels, bc, 1, bias=False) if bottleneck else None
        self.convs = nn.ModuleList([nn.Conv1d(bc, n_filters, k, padding=k//2, bias=False) for k in kernel_sizes])
        self.maxpool_conv = nn.Sequential(nn.MaxPool1d(3,1,1), nn.Conv1d(in_channels, n_filters, 1, bias=False))
        out_ch = n_filters * (len(kernel_sizes)+1)
        self.bn = nn.BatchNorm1d(out_ch); self.act = nn.ReLU()

    def forward(self, x):
        bx = self.bottleneck(x) if self.bottleneck is not None else x
        outs = [c(bx) for c in self.convs] + [self.maxpool_conv(x)]
        L = min(o.shape[-1] for o in outs)
        return self.act(self.bn(torch.cat([o[..., :L] for o in outs], dim=1)))


class InceptionTimeMultiTask(nn.Module):
    """
    InceptionTime nf=64 backbone with dual output heads.
    cls_head: classification (main task, CrossEntropy)
    reg_head: ECG interval regression (auxiliary task, MSE on z-scored targets)
    At inference only cls_head is used.
    """
    def __init__(self, in_channels=12, num_classes=3, n_filters=64,
                 depth=6, kernel_sizes=(40,20,10), n_interval=6, dropout=0.3):
        super().__init__()
        out_ch = n_filters * (len(kernel_sizes)+1)  # 256 for nf=64

        self.blocks    = nn.ModuleList()
        self.shortcuts = nn.ModuleList()
        in_ch = in_channels
        for i in range(depth):
            self.blocks.append(InceptionBlock(in_ch, n_filters, kernel_sizes))
            if i % 3 == 0:
                self.shortcuts.append(
                    nn.Sequential(nn.Conv1d(in_ch, out_ch, 1, bias=False), nn.BatchNorm1d(out_ch))
                    if in_ch != out_ch else nn.Identity()
                )
            in_ch = out_ch
        self.gap  = nn.AdaptiveAvgPool1d(1)
        self.drop = nn.Dropout(dropout)

        self.cls_head = nn.Linear(out_ch, num_classes)   # classification
        self.reg_head = nn.Linear(out_ch, n_interval)    # interval regression

    def _backbone(self, x):
        residual = x
        for i, block in enumerate(self.blocks):
            x = block(x)
            if i % 3 == 2:
                sc = self.shortcuts[i//3](residual)
                L  = min(x.shape[-1], sc.shape[-1])
                x  = torch.relu(x[..., :L] + sc[..., :L])
                residual = x
        return self.drop(self.gap(x).squeeze(-1))  # (B, 256)

    def forward(self, x):  # x: (B, 12, T)
        feat = self._backbone(x)
        return self.cls_head(feat), self.reg_head(feat)  # (B,3), (B,6)


def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
m = InceptionTimeMultiTask(IN_CHANNELS, len(CLASS_NAMES))
print(f'InceptionTimeMultiTask: {count_params(m):,} params')
print(f'  backbone: ~{count_params(m) - m.cls_head.weight.numel() - m.cls_head.bias.numel() - m.reg_head.weight.numel() - m.reg_head.bias.numel():,} params')
print(f'  cls_head: {count_params(m.cls_head):,} params | reg_head: {count_params(m.reg_head):,} params')
del m


## 4. Training utilities

In [ ]:
def stable_softmax(logits):
    x = np.asarray(logits, dtype=np.float64)
    x -= x.max(axis=1, keepdims=True)
    e = np.exp(x)
    return (e / e.sum(axis=1, keepdims=True)).astype(np.float32)

def metrics_from_probs(y_true, probs, class_names, prefix=''):
    pred = probs.argmax(axis=1)
    labels = list(range(len(class_names)))
    recalls = recall_score(y_true, pred, labels=labels, average=None, zero_division=0)
    return {
        f'{prefix}accuracy':          accuracy_score(y_true, pred),
        f'{prefix}balanced_accuracy': balanced_accuracy_score(y_true, pred),
        f'{prefix}macro_f1':          f1_score(y_true, pred, average='macro', zero_division=0),
        f'{prefix}weighted_f1':       f1_score(y_true, pred, average='weighted', zero_division=0),
        f'{prefix}macro_precision':   precision_score(y_true, pred, average='macro', zero_division=0),
        f'{prefix}macro_recall':      recall_score(y_true, pred, average='macro', zero_division=0),
        **{f'{prefix}recall_{c}': float(recalls[i]) for i,c in enumerate(class_names)},
        f'{prefix}min_class_recall':  float(recalls.min()),
        f'{prefix}macro_ovr_auc':     roc_auc_score(y_true, probs, multi_class='ovr', average='macro'),
    }

def class_weighted_ce(frame, label_col, n_classes):
    w = compute_class_weight('balanced', classes=np.arange(n_classes), y=frame[label_col].values)
    return nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32).to(DEVICE))

def run_epoch(model, loader, criterion, optimizer, scaler, training, desc):
    model.train(training)
    total_ce, total_mse, all_logits, all_labels, all_stems = 0.0, 0.0, [], [], []
    ctx = torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == 'cuda'))
    for x, ivec, y, stems in tqdm(loader, desc=desc, leave=False):
        x, ivec, y = x.to(DEVICE), ivec.to(DEVICE), y.to(DEVICE)
        with ctx:
            logits, interval_pred = model(x)          # model takes x only
            ce_loss  = criterion(logits, y)
            mse_loss = F.mse_loss(interval_pred, ivec) # both z-scored
            loss = ce_loss + LAMBDA_AUX * mse_loss
        if training:
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
        total_ce  += ce_loss.item()  * len(y)
        total_mse += mse_loss.item() * len(y)
        all_logits.append(logits.detach().cpu().float().numpy())
        all_labels.append(y.cpu().numpy())
        all_stems.extend(stems)
    N = len(loader.dataset)
    return total_ce/N, total_mse/N, np.concatenate(all_labels), np.concatenate(all_logits), np.array(all_stems)

def plot_cm(y_true, y_pred, y_probs, fold, macro_f1, auc, class_names, val_score=None):
    import matplotlib.pyplot as plt
    n = len(class_names)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(n)))
    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(n+2, n+1))
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    short = [c.split('_',1)[-1] for c in class_names]
    ax.set_xticks(range(n), short, rotation=30, ha='right')
    ax.set_yticks(range(n), short)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f'{cm[i,j]}\n{cm_norm[i,j]:.2f}', ha='center', va='center',
                    fontsize=9, color='white' if cm_norm[i,j]>0.6 else 'black')
    title = f'Fold {fold} | F1={macro_f1:.3f} AUC={auc:.3f}'
    if val_score: title += f' val={val_score:.4f}'
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout(); plt.show()


## 5. 5-fold cross-validation

In [ ]:
def train_one_fold(fold, train_df, val_df):
    fold_dir     = ARTIFACT_DIR / f'fold_{fold}'
    fold_dir.mkdir(parents=True, exist_ok=True)
    best_path    = fold_dir / 'best_model.pt'
    metrics_path = fold_dir / 'heldout_metrics.csv'
    logits_path  = fold_dir / 'heldout_logits.npz'

    if SKIP_COMPLETED_FOLDS and metrics_path.exists() and logits_path.exists() and best_path.exists():
        print(f'Fold {fold}: done — skipping.')
        return pd.read_csv(metrics_path).iloc[0].to_dict()

    seed = SEED + fold
    set_seed(seed)

    means, stds = compute_interval_stats(train_df)
    np.save(fold_dir / 'interval_stats.npy', np.stack([means, stds]))

    train_loader = make_loader(train_df,   means, stds, BATCH_SIZE, True,  True,  seed)
    val_loader   = make_loader(val_df,     means, stds, BATCH_SIZE, False, False, seed)
    held_loader  = make_loader(heldout_df, means, stds, BATCH_SIZE, False, False, seed)

    model     = InceptionTimeMultiTask(IN_CHANNELS, len(CLASS_NAMES)).to(DEVICE)
    criterion = class_weighted_ce(train_df, 'class_id', len(CLASS_NAMES))
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=MIN_LR)
    scaler    = torch.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))

    best_score, patience_ctr = -float('inf'), 0
    history = []

    for epoch in tqdm(range(1, EPOCHS+1), desc=f'fold {fold}'):
        tce, tmse, ty, tlo, _ = run_epoch(model, train_loader, criterion, optimizer, scaler, True,  f'f{fold} tr')
        vce, vmse, vy, vlo, _ = run_epoch(model, val_loader,   criterion, None,      None,   False, f'f{fold} val')
        scheduler.step()
        tp = stable_softmax(tlo); vp = stable_softmax(vlo)
        tm = metrics_from_probs(ty, tp, CLASS_NAMES, 'train_')
        vm = metrics_from_probs(vy, vp, CLASS_NAMES, 'val_')
        score = vm['val_macro_f1']
        history.append({
            'fold': fold, 'epoch': epoch,
            'train_ce_loss': tce, 'train_mse_loss': tmse,
            'val_ce_loss': vce,   'val_mse_loss': vmse,
            'best_score': score, **tm, **vm,
        })
        tqdm.write(
            f'fold={fold} ep={epoch:03d} '
            f'tce={tce:.4f} tmse={tmse:.4f} '
            f'vce={vce:.4f} vmse={vmse:.4f} '
            f'vf1={vm["val_macro_f1"]:.4f} vauc={vm["val_macro_ovr_auc"]:.4f}'
        )
        if score > best_score:
            best_score = score; patience_ctr = 0
            torch.save({
                'model_state_dict': model.state_dict(), 'fold': fold,
                'epoch': epoch, 'best_score': best_score,
                'interval_means': means, 'interval_stds': stds,
            }, best_path)
        else:
            patience_ctr += 1
            if patience_ctr >= PATIENCE:
                tqdm.write(f'fold={fold} early stop at epoch {epoch}'); break

    pd.DataFrame(history).to_csv(fold_dir / 'training_history.csv', index=False)

    # Heldout evaluation
    ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    best_epoch = ckpt['epoch']

    model.eval()
    all_logits, all_labels, all_stems = [], [], []
    ctx = torch.amp.autocast(device_type=DEVICE.type, enabled=(DEVICE.type == 'cuda'))
    with torch.no_grad():
        for x, ivec, y, stems in tqdm(held_loader, desc=f'f{fold} heldout', leave=False):
            x = x.to(DEVICE)
            with ctx:
                logits, _ = model(x)  # classification head only
            all_logits.append(logits.cpu().float().numpy())
            all_labels.append(y.numpy())
            all_stems.extend(stems)
    hl = np.concatenate(all_logits); hy = np.concatenate(all_labels)
    hp = stable_softmax(hl)
    hm = metrics_from_probs(hy, hp, CLASS_NAMES, 'heldout_')

    plot_cm(hy, hp.argmax(1), hp, fold, hm['heldout_macro_f1'], hm['heldout_macro_ovr_auc'],
            CLASS_NAMES, val_score=best_score)

    val_metrics = history[-1] if not history[-1].get('best_score') else \
        max(history, key=lambda r: r['best_score'])
    row = {**hm, 'fold': fold, 'best_epoch': best_epoch, 'best_score': best_score}
    pd.DataFrame([row]).to_csv(metrics_path, index=False)
    np.savez(logits_path, probs=hp, y_true=hy, stems=np.array(all_stems))

    train_df.to_csv(fold_dir / 'train_split.csv', index=False)
    val_df.to_csv(fold_dir   / 'val_split.csv',   index=False)

    print(f'Fold {fold} | heldout F1={hm["heldout_macro_f1"]:.4f} AUC={hm["heldout_macro_ovr_auc"]:.4f} | best_ep={best_epoch}')
    return row


In [ ]:
set_seed(SEED)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_results = []

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_val_df, train_val_df['class_id'])):
    train_df = train_val_df.iloc[tr_idx].copy().reset_index(drop=True)
    val_df   = train_val_df.iloc[va_idx].copy().reset_index(drop=True)
    print(f'\nFold {fold}: train={len(train_df)}, val={len(val_df)}')
    result = train_one_fold(fold, train_df, val_df)
    fold_results.append(result)

results_df = pd.DataFrame(fold_results)
print('\n=== Per-fold heldout ===')
cols = ['fold', 'heldout_macro_f1', 'heldout_macro_ovr_auc',
        'heldout_recall_0_control', 'heldout_recall_1_significant_mr',
        'heldout_recall_2_significant_ms', 'best_epoch', 'best_score']
print(results_df[cols].to_string(index=False))
results_df.to_csv(ARTIFACT_DIR / 'heldout_fold_metrics.csv', index=False)


## 6. 5-fold soft ensemble

In [ ]:
probs_list, y_ref, stems_ref = [], None, None
for fold in range(5):
    d = np.load(ARTIFACT_DIR / f'fold_{fold}/heldout_logits.npz')
    probs_list.append(d['probs'])
    if y_ref is None:
        y_ref = d['y_true'].astype(int)
        stems_ref = d['stems']

# Soft ensemble
soft_probs = np.mean(probs_list, axis=0)
soft_preds = soft_probs.argmax(axis=1)
em = metrics_from_probs(y_ref, soft_probs, CLASS_NAMES, 'heldout_')
em['n_models'] = 5

print('=== 5-fold soft ensemble ===')
print(f"  Macro F1   : {em['heldout_macro_f1']:.4f}")
print(f"  AUC        : {em['heldout_macro_ovr_auc']:.4f}")
print(f"  Ctrl recall: {em['heldout_recall_0_control']:.4f}")
print(f"  MR recall  : {em['heldout_recall_1_significant_mr']:.4f}")
print(f"  MS recall  : {em['heldout_recall_2_significant_ms']:.4f}")

pd.DataFrame([em]).to_csv(ARTIFACT_DIR / 'heldout_ensemble_metrics.csv', index=False)

# Confusion matrix
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_ref, soft_preds)
cm_norm = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(5, 4))
ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
short = [c.split('_',1)[-1] for c in CLASS_NAMES]
ax.set_xticks(range(3), short, rotation=30, ha='right')
ax.set_yticks(range(3), short)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{cm[i,j]}\n{cm_norm[i,j]:.2f}', ha='center', va='center',
                fontsize=9, color='white' if cm_norm[i,j]>0.6 else 'black')
ax.set_title(f"Soft ensemble | F1={em['heldout_macro_f1']:.4f} AUC={em['heldout_macro_ovr_auc']:.4f}")
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout(); plt.show()
